In [19]:
import os
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_kws_data_fixed(data_path='recordings/'):
    x_data = []
    y_labels = []
    
    for filename in os.listdir(data_path):
        if filename.endswith('.wav'):
            label = filename.split('_')[0]
            path = os.path.join(data_path, filename)
            
            # Load and Normalize Audio
            audio, sr = librosa.load(path, sr=16000, duration=1.0)
            audio = librosa.util.fix_length(audio, size=16000)
            
            # Feature Extraction: 40 MFCCs
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            
            # Normalization: MFCCs can have large values; scaling helps quantization
            mfcc = (mfcc - np.mean(mfcc)) / np.std(mfcc)
            
            x_data.append(mfcc)
            y_labels.append(label)
            
    return np.array(x_data), np.array(y_labels)

# 1. Load Data
X, y = load_kws_data_fixed()
X = X[..., np.newaxis] # Shape: (samples, 40, 32, 1)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [20]:
model_kws = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(40, 32, 1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(le.classes_), activation='softmax')
])

model_kws.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_kws.fit(X_train, y_train, epochs=30, validation_data=(X_test, y_test))

Epoch 1/30


c:\Users\ng822\anaconda3\envs\421\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.3217 - loss: 1.8917 - val_accuracy: 0.6217 - val_loss: 1.1376
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7688 - loss: 0.7219 - val_accuracy: 0.8900 - val_loss: 0.4314
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8975 - loss: 0.3582 - val_accuracy: 0.9333 - val_loss: 0.2553
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9400 - loss: 0.2106 - val_accuracy: 0.9517 - val_loss: 0.1907
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9546 - loss: 0.1502 - val_accuracy: 0.9550 - val_loss: 0.1572
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9675 - loss: 0.1111 - val_accuracy: 0.9733 - val_loss: 0.1129
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9754 - loss: 0.0917 - val_accuracy: 0.9633 - val_loss: 0.1110
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9787 - loss: 0.0790 - val_accuracy: 0.9800 - val_loss: 0.

In [21]:
# Updated Representative Generator for your 3000-item dataset
def representative_data_gen():
    for i in range(300): # Increased for better calibration
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model_kws)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_kws = converter.convert()
with open('kws_model.tflite', 'wb') as f:
    f.write(tflite_kws)

INFO:tensorflow:Assets written to: C:\Users\ng822\AppData\Local\Temp\tmpgo5lao0f\assets


INFO:tensorflow:Assets written to: C:\Users\ng822\AppData\Local\Temp\tmpgo5lao0f\assets


Saved artifact at 'C:\Users\ng822\AppData\Local\Temp\tmpgo5lao0f'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='keras_tensor_25')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2139927900624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139927892752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926845712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926846480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926846096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926846672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926845904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2139926848400: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ng822\anaconda3\envs\421\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [22]:
def write_to_h(data, name):
    hex_data = [f'0x{b:02x}' for b in data]
    with open(f'{name}.h', 'w') as f:
        f.write(f'unsigned char {name}[] = {{\n' + ', '.join(hex_data) + '\n};\n')
        f.write(f'unsigned int {name}_len = {len(data)};')

write_to_h(tflite_kws, 'kws_model_data')
print("Section 12.8 Complete: kws_model_data.h generated.")

Section 12.8 Complete: kws_model_data.h generated.


In [23]:
import numpy as np
import tensorflow as tf

def evaluate_tflite_batch(model_path, X_test, y_test):
    # 1. Load the TFLite model and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    # 2. Get input and output details
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    
    # 3. Get quantization parameters (Scale and Zero Point)
    input_scale, input_zero_point = input_details['quantization']
    
    predictions = []
    
    print(f"Starting batch evaluation on {len(X_test)} samples...")

    for i in range(len(X_test)):
        # Get a single sample and add batch dimension if necessary
        sample = X_test[i]
        if len(sample.shape) == 3: # (40, 32, 1) -> (1, 40, 32, 1)
            sample = np.expand_dims(sample, axis=0)

        # 4. MANUALLY QUANTIZE: Float32 -> Int8
        # Formula: value_int8 = (value_float / scale) + zero_point
        sample_int8 = (sample / input_scale + input_zero_point).astype(np.int8)

        # 5. Set the tensor and invoke
        interpreter.set_tensor(input_details['index'], sample_int8)
        interpreter.invoke()

        # 6. Get result and de-quantize if necessary (or just take argmax)
        output = interpreter.get_tensor(output_details['index'])
        predictions.append(np.argmax(output))

    # 7. Calculate Accuracy
    predictions = np.array(predictions)
    accuracy = np.mean(predictions == y_test)
    
    return accuracy

# Run the evaluation
tflite_acc = evaluate_tflite_batch("kws_model.tflite", X_test, y_test)
print(f"Batch TFLite Accuracy: {tflite_acc * 100:.2f}%")

Starting batch evaluation on 600 samples...


c:\Users\ng822\anaconda3\envs\421\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Batch TFLite Accuracy: 96.17%
